# Criação de Mapa Interativo
---
INTEGRANTES:
- Giovanna Coelho Gilio de Sá
- Gustavo Gomes Jacinto

FERRAMENTAS:
- Folium
- Python
- Base de Dados Pública
- API SIDRA
- Pandas

### 1.0 Instalação Folium

In [2]:
!pip install folium


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2.0 Import das Bibliotecas

In [3]:
import pandas as pd
import folium
from folium.plugins import GroupedLayerControl

### 3.0 Análise das Bases de Dados

| Acesso a base de dados publicas encontradas no [GitHub](https://github.com/alanwillms/geoinfo), onde os dados originalmente foram extraídos do FTP do IBGE.

Passos:
- Leitura dos arquivos `.csv`
- Análise das primeiras linhas do DataFrame
- Junção do df_municipios e df_estados a partir do codigo_uf
- Filtro das capitais brasileiras
- Renomeando algumas colunas
- Junção/Concatenação de latitude + longitude, formando uma coordenada
- Requisição da API publica do IBGE SIDRA
- Requisição de dados populacionais e colocando em um DataFrame
- Junção do DataFrame df_capitais + df_pop
- Criação do Mapa

### 4.0 Exploração Inicial

In [4]:
df_estados = pd.read_csv("estados.csv")
df_municipios = pd.read_csv("municipios.csv")

In [5]:
df_estados.head()

,codigo_uf,uf,nome,latitude,longitude,regiao
0,11,RO,Rondônia,-10.83,-63.34,Norte
1,12,AC,Acre,-8.77,-70.55,Norte
2,13,AM,Amazonas,-3.47,-65.10,Norte
3,14,RR,Roraima,1.99,-61.33,Norte
4,15,PA,Pará,-3.79,-52.48,Norte


In [6]:
df_municipios.head()

,codigo_ibge,nome,latitude,longitude,capital,codigo_uf,siafi_id,ddd,fuso_horario
0,5200050,Abadia de Goiás,-16.75730,-49.4412,0,52,1050,62,America/Sao_Paulo
1,3100104,Abadia dos Dourados,-18.48310,-47.3916,0,31,4001,34,America/Sao_Paulo
2,5200100,Abadiânia,-16.19700,-48.7057,0,52,9201,62,America/Sao_Paulo
3,3100203,Abaeté,-19.15510,-45.4444,0,31,4003,37,America/Sao_Paulo
4,1500107,Abaetetuba,-1.72183,-48.8788,0,15,401,91,America/Sao_Paulo


### 5.0 Junções, Correções, Filtros nas Tabelas

#### 5.1 Junção dos DataFrames

In [7]:
df = pd.merge(
    df_municipios,
    df_estados,
    on="codigo_uf",
    how="left"
)

#### 5.2 Renomeando Colunas do DataFrame

In [8]:
df = df.rename(columns={
    "nome_x": "municipio",
    "nome_y": "estado",
    "latitude_x": "lat",
    "longitude_x": "lon"
})

In [9]:
df.head()

,codigo_ibge,municipio,lat,lon,capital,codigo_uf,siafi_id,ddd,fuso_horario,uf,estado,latitude_y,longitude_y,regiao
0,5200050,Abadia de Goiás,-16.75730,-49.4412,0,52,1050,62,America/Sao_Paulo,GO,Goiás,-15.98,-49.86,Centro-Oeste
1,3100104,Abadia dos Dourados,-18.48310,-47.3916,0,31,4001,34,America/Sao_Paulo,MG,Minas Gerais,-18.10,-44.38,Sudeste
2,5200100,Abadiânia,-16.19700,-48.7057,0,52,9201,62,America/Sao_Paulo,GO,Goiás,-15.98,-49.86,Centro-Oeste
3,3100203,Abaeté,-19.15510,-45.4444,0,31,4003,37,America/Sao_Paulo,MG,Minas Gerais,-18.10,-44.38,Sudeste
4,1500107,Abaetetuba,-1.72183,-48.8788,0,15,401,91,America/Sao_Paulo,PA,Pará,-3.79,-52.48,Norte


#### 5.3 Filtro de Capitais brasileiras

In [10]:
df_capitais = df[df["capital"] == 1]

In [11]:
df_capitais["coordenadas"] = (
    df_capitais["lat"].astype(str) + ", " + df_capitais["lon"].astype(str)
)

In [12]:
df_capitais.head(27)

,codigo_ibge,municipio,lat,lon,capital,codigo_uf,siafi_id,ddd,fuso_horario,uf,estado,latitude_y,longitude_y,regiao,coordenadas
294,2800308,Aracaju,-10.909100,-37.0677,1,28,3105,79,America/Sao_Paulo,SE,Sergipe,-10.57,-37.45,Nordeste,"-10.9091, -37.0677"
580,1501402,Belém,-1.455400,-48.4898,1,15,427,91,America/Sao_Paulo,PA,Pará,-3.79,-52.48,Norte,"-1.4554, -48.4898"
592,3106200,Belo Horizonte,-19.910200,-43.9266,1,31,4123,31,America/Sao_Paulo,MG,Minas Gerais,-18.10,-44.38,Sudeste,"-19.9102, -43.9266"
642,1400100,Boa Vista,2.823840,-60.6753,1,14,301,95,America/Porto_Velho,RR,Roraima,1.99,-61.33,Norte,"2.82384, -60.6753"
755,5300108,Brasília,-15.779500,-47.9297,1,53,9701,61,America/Sao_Paulo,DF,Distrito Federal,-15.83,-47.86,Centro-Oeste,"-15.7795, -47.9297"
972,5002704,Campo Grande,-20.448600,-54.6295,1,50,9051,67,America/Porto_Velho,MS,Mato Grosso do Sul,-20.51,-54.54,Centro-Oeste,"-20.4486, -54.6295"
1491,5103403,Cuiabá,-15.601000,-56.0974,1,51,9067,65,America/Porto_Velho,MT,Mato Grosso,-12.64,-55.42,Centro-Oeste,"-15.601, -56.0974"
1508,4106902,Curitiba,-25.419500,-49.2646,1,41,7535,41,America/Sao_Paulo,PR,Paraná,-24.89,-51.55,Sul,"-25.4195, -49.2646"
1812,4205407,Florianópolis,-27.594500,-48.5477,1,42,8105,48,America/Sao_Paulo,SC,Santa Catarina,-27.45,-50.95,Sul,"-27.5945, -48.5477"
1831,2304400,Fortaleza,-3.716640,-38.5423,1,23,1389,85,America/Sao_Paulo,CE,Ceará,-5.20,-39.53,Nordeste,"-3.71664, -38.5423"


### 6.0 Aplicando API para dados demográficos

#### 6.1 Criando requisição para API

In [13]:
import requests

url = "https://servicodados.ibge.gov.br/api/v3/agregados/4714/periodos/-6/variaveis/93?localidades=N3[all]"

res = requests.get(url)
dados = res.json()

dados[0]['resultados'][0]['series'][:3]

[{'localidade': {'id': '11',
   'nivel': {'id': 'N3', 'nome': 'Unidade da Federação'},
   'nome': 'Rondônia'},
  'serie': {'2022': '1581196'}},
 {'localidade': {'id': '12',
   'nivel': {'id': 'N3', 'nome': 'Unidade da Federação'},
   'nome': 'Acre'},
  'serie': {'2022': '830018'}},
 {'localidade': {'id': '13',
   'nivel': {'id': 'N3', 'nome': 'Unidade da Federação'},
   'nome': 'Amazonas'},
  'serie': {'2022': '3941613'}}]

#### 6.2 Limpeza e Estruturação dos Dados

In [14]:
lista = []

for item in dados[0]['resultados'][0]['series']:
    estado = item['localidade']['nome']
    populacao = list(item['serie'].values())[0]

    lista.append({
        "estado": estado,
        "populacao": int(populacao)
    })

df_pop = pd.DataFrame(lista)

In [15]:
print (df_pop)

                 estado  populacao
0              Rondônia    1581196
1                  Acre     830018
2              Amazonas    3941613
3               Roraima     636707
4                  Pará    8120131
5                 Amapá     733759
6             Tocantins    1511460
7              Maranhão    6776699
8                 Piauí    3271199
9                 Ceará    8794957
10  Rio Grande do Norte    3302729
11              Paraíba    3974687
12           Pernambuco    9058931
13              Alagoas    3127683
14              Sergipe    2210004
15                Bahia   14141626
16         Minas Gerais   20539989
17       Espírito Santo    3833712
18       Rio de Janeiro   16055174
19            São Paulo   44411238
20               Paraná   11444380
21       Santa Catarina    7610361
22    Rio Grande do Sul   10882965
23   Mato Grosso do Sul    2757013
24          Mato Grosso    3658649
25                Goiás    7056495
26     Distrito Federal    2817381


#### 6.3 Junção do df_capitais com df_pop

In [16]:
df_final = df_capitais.merge(df_pop, on="estado")

In [17]:
df_final

,codigo_ibge,municipio,lat,lon,capital,codigo_uf,siafi_id,ddd,fuso_horario,uf,estado,latitude_y,longitude_y,regiao,coordenadas,populacao
0,2800308,Aracaju,-10.909100,-37.0677,1,28,3105,79,America/Sao_Paulo,SE,Sergipe,-10.57,-37.45,Nordeste,"-10.9091, -37.0677",2210004
1,1501402,Belém,-1.455400,-48.4898,1,15,427,91,America/Sao_Paulo,PA,Pará,-3.79,-52.48,Norte,"-1.4554, -48.4898",8120131
2,3106200,Belo Horizonte,-19.910200,-43.9266,1,31,4123,31,America/Sao_Paulo,MG,Minas Gerais,-18.10,-44.38,Sudeste,"-19.9102, -43.9266",20539989
3,1400100,Boa Vista,2.823840,-60.6753,1,14,301,95,America/Porto_Velho,RR,Roraima,1.99,-61.33,Norte,"2.82384, -60.6753",636707
4,5300108,Brasília,-15.779500,-47.9297,1,53,9701,61,America/Sao_Paulo,DF,Distrito Federal,-15.83,-47.86,Centro-Oeste,"-15.7795, -47.9297",2817381
5,5002704,Campo Grande,-20.448600,-54.6295,1,50,9051,67,America/Porto_Velho,MS,Mato Grosso do Sul,-20.51,-54.54,Centro-Oeste,"-20.4486, -54.6295",2757013
6,5103403,Cuiabá,-15.601000,-56.0974,1,51,9067,65,America/Porto_Velho,MT,Mato Grosso,-12.64,-55.42,Centro-Oeste,"-15.601, -56.0974",3658649
7,4106902,Curitiba,-25.419500,-49.2646,1,41,7535,41,America/Sao_Paulo,PR,Paraná,-24.89,-51.55,Sul,"-25.4195, -49.2646",11444380
8,4205407,Florianópolis,-27.594500,-48.5477,1,42,8105,48,America/Sao_Paulo,SC,Santa Catarina,-27.45,-50.95,Sul,"-27.5945, -48.5477",7610361
9,2304400,Fortaleza,-3.716640,-38.5423,1,23,1389,85,America/Sao_Paulo,CE,Ceará,-5.20,-39.53,Nordeste,"-3.71664, -38.5423",8794957


### 7.0 Criação do Mapa de Capitais Interativo

In [18]:
# criação do mapa
brasil = folium.Map(location=[-27.5945, -48.5477], zoom_start=4, control_scale=True)
folium.TileLayer('cartodbpositron').add_to(brasil)

# definição de cores
cores = {
    "Norte": "green",
    "Nordeste": "red",
    "Centro-Oeste": "blue",
    "Sudeste": "purple",
    "Sul": "orange"
}

# grupo de marcadores
grupos = {}

for i, linha in df_final.iterrows():
    regiao = linha['regiao'] # pega região

    # verificando se regiao possui um grupo
    if regiao not in grupos:
        grupos[regiao] = folium.FeatureGroup(name=regiao)

    # criando marcadires e pop up
    folium.Marker(
        location=[linha['lat'], linha['lon']],
        popup = folium.Popup(
            f"{linha['municipio']}<br> <br>População do Estado: {linha['populacao']:,}", 
            max_width=300,  
            min_width=150
        ),
        icon=folium.Icon(
            color=cores[linha['regiao']],
            icon = "glyphicon glyphicon-heart-empty",   
            icon_color = "white",
            prefix = "glyphicon"            
        )
    ).add_to(grupos[regiao])

for grupo in grupos.values():
    grupo.add_to(brasil)

# controle de "camadas" ou seja o filtro
folium.LayerControl().add_to(brasil)

brasil

In [19]:
brasil.save("mapa_brasil.html")